# Parda — web app M4a: browser OCR (Tesseract.js) on both benchmarks

The web app reads pages with **Tesseract.js** (Tesseract compiled for the browser), not the Tesseract the benchmarks
used. This runs the app's own OCR module (`web/src/ocr`) on the 300 seen + 300 unseen benchmark pages and measures
(1) how well it reads the true text and the PII, and (2) the **end-to-end** result with the shipped ONNX models.

**Settings:** Accelerator **None (CPU)** · Internet **On**
**Inputs (Add Input):** Your Work → **Parda** (Phase 1) · **08b** · **10** (compression) · Datasets tab → `celeba-dataset`
**Secrets:** `GITHUB_TOKEN`, `HF_TOKEN`. Run cells 1–4 by hand (~10 min), then **Save Version → Save & Run All** (~2 h).


In [ ]:
# 1. Code + secrets
import os, subprocess
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
GH = secrets.get_secret("GITHUB_TOKEN")
os.environ["HF_TOKEN"] = secrets.get_secret("HF_TOKEN")
REPO, CODE = "chetanjagan/Parda", "/tmp/Parda"
if not os.path.exists(CODE):
    subprocess.run(["git", "clone", "-q", f"https://{GH}@github.com/{REPO}.git", CODE], check=True)
    subprocess.run(["git", "-C", CODE, "remote", "set-url", "origin", f"https://github.com/{REPO}.git"], check=True)
else:
    subprocess.run(["git", "-C", CODE, "pull", "-q", f"https://{GH}@github.com/{REPO}.git", "main"], check=True)
os.chdir(CODE)
print(subprocess.run(["git", "log", "--oneline", "-3"], capture_output=True, text=True).stdout)


In [ ]:
# 2. Fonts (to regenerate the unseen pages), Python packages, and Node.js 22 for the web code
!bash scripts/setup_kaggle.sh
!pip install -q gliner ultralytics onnxruntime rapidfuzz
!curl -fsSL https://nodejs.org/dist/v22.12.0/node-v22.12.0-linux-x64.tar.xz -o /tmp/node.tar.xz && tar -xJf /tmp/node.tar.xz -C /tmp && ln -sf /tmp/node-v22.12.0-linux-x64/bin/* /usr/local/bin/
!node -v && npm -v


In [ ]:
# 3. Build the web code (npm install brings Tesseract.js) and run its OCR tests
!cd web && npm install --no-audit --no-fund --loglevel=error && npx tsc -p tsconfig.json && node --test test/ocr.test.mjs 2>&1 | grep -E "^ok|^not ok|^# (pass|fail)"


In [ ]:
# 4. Inputs: data, faces, saved OCR + v2 reports (08b), native browser-build reports (10), shipped ONNX models
import json, shutil
from huggingface_hub import HfApi, snapshot_download
from parda.fsutil import find_path
from parda.ocr.run_ocr import find_data, _n_images
from parda.vision.faces import find_faces_dir
IN, W = "/kaggle/input", "/kaggle/working"
first = lambda *rels: next((h[0] for r in rels for h in [find_path(IN, r)] if h), None)
DATA = find_data()
FACES = find_faces_dir(IN, exclude=[DATA])
C_SEEN = os.path.dirname(first("cache_seen/ocr_easyocr.jsonl") or "")
C_OOD = os.path.dirname(first("cache_ood/ocr_easyocr.jsonl") or "")
N_SEEN, N_OOD = first("compress_results/report_seen.json"), first("compress_results/report_unseen.json")
HF_USER = HfApi(token=os.environ["HF_TOKEN"]).whoami()["name"]
snapshot_download(f"{HF_USER}/parda-onnx-v1", local_dir=f"{W}/onnx", token=os.environ["HF_TOKEN"],
                  allow_patterns=["gliner/model_int8_embed.onnx", "gliner/parda_onnx.json", "yolo/parda-yolo.onnx"])
for k, v in [("DATA", DATA), ("FACES", FACES), ("OCR seen (native)", C_SEEN), ("OCR unseen (native)", C_OOD),
             ("native browser report seen", N_SEEN), ("native browser report unseen", N_OOD)]:
    print(f"{k:30s} {v}")
assert DATA and _n_images(DATA) == 30000 and FACES and C_SEEN and C_OOD, "attach Parda, 08b and celeba-dataset"
if not (N_SEEN and N_OOD):
    print("NOTE: notebook 10 not attached -> the comparison uses the numbers recorded in results/onnx instead")


In [ ]:
# 5. The 300 seen pages (real faces, as in Phase 4/5) and the 300 unseen phone-photo pages, as page lists
from parda.pipeline.evaluate_e2e import load_pages
!python -m parda.vision.yolo_data --data {DATA} --out {W}/yolo_bench --faces_dir {FACES} --only bench --workers 4 > /dev/null
!python -m parda.synth.ood --n 300 --out {W}/ood --faces_dir {FACES} --workers 4 > /dev/null
def write_pages(pages, path):
    with open(path, "w") as f:
        for p in pages:
            f.write(json.dumps({"id": p["id"], "path": p["path"], "lang": p["rec"]["lang"]}) + "\n")
    return len(pages)
print("seen:", write_pages(load_pages(DATA, f"{W}/yolo_bench"), f"{W}/pages_seen.jsonl"),
      "| unseen:", write_pages(load_pages(f"{W}/ood"), f"{W}/pages_unseen.jsonl"))


In [ ]:
# 6. BROWSER OCR: Tesseract.js (the app's own module) on all 600 pages (~30-40 min; resumable)
!cd web && node tools/ocr_pages.mjs --pages {W}/pages_seen.jsonl --out {W}/tjs_seen.jsonl --workers 4
!cd web && node tools/ocr_pages.mjs --pages {W}/pages_unseen.jsonl --out {W}/tjs_unseen.jsonl --workers 4
for f in ("tjs_seen.jsonl", "tjs_unseen.jsonl"):
    rows = [json.loads(l) for l in open(f"{W}/{f}")]
    print(f, len(rows), "pages |", sum(1 for r in rows if r.get("error")), "errors |", sum(len(r["segments"]) for r in rows), "words")


In [ ]:
# 7. How well each OCR reads the TRUE text and the PII (same scoring as the Phase 2 OCR benchmark)
from IPython.display import Markdown, display
!python tools/compare_ocr.py --data {DATA} --ocr "native Tesseract={C_SEEN}/ocr_tesseract.jsonl" --ocr "Tesseract.js={W}/tjs_seen.jsonl" --title "Seen templates" --out {W}/ocr_seen.md > /dev/null
!python tools/compare_ocr.py --data {W}/ood --ocr "native Tesseract={C_OOD}/ocr_tesseract.jsonl" --ocr "Tesseract.js={W}/tjs_unseen.jsonl" --title "Unseen layouts + phone photos" --out {W}/ocr_unseen.md > /dev/null
display(Markdown(open(f"{W}/ocr_seen.md").read() + "\n" + open(f"{W}/ocr_unseen.md").read()))


In [ ]:
# 8. END TO END with browser OCR (shipped ONNX models; EasyOCR rows use the saved native EasyOCR) (~70 min)
for bench, cdir, tjs in (("seen", C_SEEN, "tjs_seen.jsonl"), ("unseen", C_OOD, "tjs_unseen.jsonl")):
    os.makedirs(f"{W}/cache_tjs_{bench}", exist_ok=True)
    shutil.copy(f"{W}/{tjs}", f"{W}/cache_tjs_{bench}/ocr_tesseract.jsonl")
    shutil.copy(f"{cdir}/ocr_easyocr.jsonl", f"{W}/cache_tjs_{bench}/ocr_easyocr.jsonl")
!python -m parda.pipeline.evaluate_e2e --data {DATA} --yolo_data {W}/yolo_bench --ocr tesseract,easyocr --gliner_ft {W}/onnx/gliner --yolo {W}/onnx/yolo/parda-yolo.onnx --cache {W}/cache_tjs_seen --out {W}/tjs_e2e_seen --title "Browser OCR (Tesseract.js) — seen templates" 2>&1 | grep -v -i warning | tail -8
!python -m parda.pipeline.evaluate_e2e --data {W}/ood --ocr tesseract,easyocr --gliner_ft {W}/onnx/gliner --yolo {W}/onnx/yolo/parda-yolo.onnx --cache {W}/cache_tjs_unseen --out {W}/tjs_e2e_unseen --title "Browser OCR (Tesseract.js) — unseen layouts + phone photos" 2>&1 | grep -v -i warning | tail -8


In [ ]:
# 9. Native Tesseract vs Tesseract.js, end to end (same shipped models)
pct = lambda v: "–" if v is None else f"{100 * v:.1f}%"
NATIVE_FALLBACK = {"seen": {"browser build: Tesseract only + YOLO": 0.897, "+ YOLO visual = Parda": 0.972},
                   "unseen": {"browser build: Tesseract only + YOLO": 0.676, "+ YOLO visual = Parda": 0.874}}  # results/onnx
rows = ["| Benchmark | Configuration | All PII fully hidden: native Tesseract → Tesseract.js | Pages with no leak (Tesseract.js) |",
        "|---|---|---|---|"]
for bench, native_path in (("seen", N_SEEN), ("unseen", N_OOD)):
    tj = {r["config"]: r for r in json.load(open(f"{W}/tjs_e2e_{bench}/report_e2e.json"))["configs"]}
    nat = {r["config"]: r["fully_redacted"] for r in json.load(open(native_path))["configs"]} if native_path else NATIVE_FALLBACK[bench]
    for name in ("fine-tuned GLiNER + rules", "browser build: Tesseract only + YOLO", "+ YOLO visual = Parda"):
        if name in tj:
            rows.append(f"| {bench} | {name} | {pct(nat.get(name))} → **{pct(tj[name]['fully_redacted'])}** | {pct(tj[name]['pages_clean'])} |")
report = "\n".join(["### Browser OCR end to end", "", *rows, "",
                    "Rows with EasyOCR use the saved native EasyOCR text (EasyOCR in the browser is milestone M4b)."]) + "\n"
open(f"{W}/tjs_vs_native.md", "w").write(report)
display(Markdown(report))


In [ ]:
# 10. Collect results (download from the Output tab)
R = f"{W}/browser_ocr_results"
os.makedirs(R, exist_ok=True)
for f in ("tjs_vs_native.md", "ocr_seen.md", "ocr_seen.json", "ocr_unseen.md", "ocr_unseen.json"):
    shutil.copy(f"{W}/{f}", f"{R}/{f}")
for b in ("seen", "unseen"):
    for ext in ("md", "json"):
        shutil.copy(f"{W}/tjs_e2e_{b}/report_e2e.{ext}", f"{R}/report_{b}.{ext}")
print(sorted(os.listdir(R)))
